In [58]:
#pip install -U langchain
#pip install chromadb -U
#pip install pandas -U
#pip install -U pypdf

In [2]:
import chromadb
from chromadb.utils import embedding_functions

# 1. create vectore store using chromadb.Client(), use cosine distance

In [3]:
chroma_client = chromadb.Client()
embedding_func = embedding_functions.DefaultEmbeddingFunction()
# create collection using cosine distance 
chroma_collection = chroma_client.get_or_create_collection("chroma_movies",
                                                            embedding_function=embedding_func,
                                                            metadata={"dnsw:space": "cosine"}
                                                            )


# 2. Process "Public Criterion Master List.csv"
- add proccessed results to the chroma store 
- include each field in metadata and source "source:csv"

In [11]:
import pandas as pd
df = pd.read_csv("./dataset/Public Criterion Master List.csv")

documents = []
metadatas = [] 
ids = []

for index, row in df.iterrows():
    # include fields
    page_content = "Spine: " + str(row["SPINE"]) + " " + \
                   "Title: " + str(row["TITLE (Year)"]) + " " + \
                   "Director: " + str(row["DIRECTOR"]) + " " + \
                   "Country: " + str(row["COUNTRY"]) + " " + \
                   "Description: " + str(row["Brief Description"])
    
    # construct metadata dictionary using all the CSV headers
    metadata_dict = {
        "spine": str(row["SPINE"]),
        "title_year": str(row["TITLE (Year)"]),
        "director": str(row["DIRECTOR"]),
        "country": str(row["COUNTRY"]),
        "4k_release": str(row["4K Release?"]),
        "imdb_rating": str(row["IMDb Rating"]),
        "rotten_tomatoes": str(row["Rotten Tomatoes"]),
        "brief_description": str(row["Brief Description"]),
        "count_sort": str(row["Count (Sort)"]),
        "year": str(row["YEAR"]),
        "source": "csv" 
    }
    
    ids.append(str(index))
    documents.append(page_content)
    metadatas.append(metadata_dict) 

# add to chroma_collection
chroma_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# 3. Query and tailor the inputs to produce 10 results including documents, metadata, embeddings, and distance values.
- print results excluding embeddings

In [12]:
query = "Which of theses movies deal with war in some way?"
query_results = chroma_collection.query(
    query_texts=[query],
    n_results=10,
    include=["documents", "metadatas", "embeddings", "distances"]
    )
# print results excludding embeddings
for idx, doc in enumerate(query_results['documents'][0]):
    print(f"Result {idx+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {query_results['metadatas'][0][idx]}")
    print(f"Distance: {query_results['distances'][0][idx]}")
    print("\n")


Result 1:
Document: Spine: 16 Title: Samurai III: Duel at Ganryu Island (1956) Director: Hiroshi Inagaki Country: Japan Description: The conclusion of the trilogy, culminating in Musashi's final, legendary duel.
Metadata: {'year': '1956.0', 'brief_description': "The conclusion of the trilogy, culminating in Musashi's final, legendary duel.", 'spine': '16', '4k_release': 'False', 'source': 'csv', 'director': 'Hiroshi Inagaki', 'country': 'Japan', 'title_year': 'Samurai III: Duel at Ganryu Island (1956)', 'rotten_tomatoes': 'nan', 'imdb_rating': '7.7/10', 'count_sort': '16'}
Distance: 1.2984875440597534


Result 2:
Document: Spine: 79 Title: W. C. Fields—Six Short Films (1933–1937) Director: Various Country: United States Description: A collection of W. C. Fields's hilarious and influential short films.
Metadata: {'title_year': 'W. C. Fields—Six Short Films (1933–1937)', 'spine': '79', '4k_release': 'False', 'source': 'csv', 'brief_description': "A collection of W. C. Fields's hilarious 

# 4. Modify the query by adding filtering.
- try "4k Release?" = True

In [21]:
query = "Which of theses movies deal with war in some way?"
query_results = chroma_collection.query(
    query_texts=[query],
    n_results=10,
    where={"4k_release": "True"},
    include=["documents", "metadatas", "embeddings", "distances"]
    )

# print results excludding embeddings
for idx, doc in enumerate(query_results['documents'][0]):
    print(f"Result {idx+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {query_results['metadatas'][0][idx]}")
    print(f"Distance: {query_results['distances'][0][idx]}")
    print("\n")


Result 1:
Document: Spine: 52 Title: Yojimbo (1961) Director: Akira Kurosawa Country: Japan Description: A masterless samurai enters a small town divided by two warring criminal gangs and pits them against each other for his own gain.
Metadata: {'spine': '52', 'year': '1961.0', 'count_sort': '52', 'imdb_rating': '8.2/10', 'title_year': 'Yojimbo (1961)', 'director': 'Akira Kurosawa', 'country': 'Japan', 'rotten_tomatoes': '97%', 'source': 'csv', '4k_release': 'True', 'brief_description': 'A masterless samurai enters a small town divided by two warring criminal gangs and pits them against each other for his own gain.'}
Distance: 1.354796051979065


Result 2:
Document: Spine: 2 Title: Seven Samurai (1954) Director: Akira Kurosawa Country: Japan Description: A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.
Metadata: {'brief_description': 'A poor village hires seven desperate but experienced samurai to protect them from 

# 5. Use get to retreive records with ids 10-12
- use the same filtering as task 4

In [32]:
results = chroma_collection.get(ids=["10", "11", "12"], include=["documents", "metadatas", "embeddings"])
for idx, doc in enumerate(results['documents']):
    print(f"Result {idx+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {results['metadatas'][idx]}")
    print("\n")

Result 1:
Document: Spine: 11 Title: The Seventh Seal (1957) Director: Ingmar Bergman Country: Sweden Description: A knight returning from the Crusades challenges Death to a game of chess to save his life and find meaning in a plague-ravaged world.
Metadata: {'director': 'Ingmar Bergman', 'spine': '11', 'brief_description': 'A knight returning from the Crusades challenges Death to a game of chess to save his life and find meaning in a plague-ravaged world.', 'imdb_rating': '8.1/10', 'count_sort': '11', '4k_release': 'True', 'source': 'csv', 'title_year': 'The Seventh Seal (1957)', 'year': '1957.0', 'rotten_tomatoes': '94%', 'country': 'Sweden'}


Result 2:
Document: Spine: 12 Title: This Is Spinal Tap (1984) Director: Rob Reiner Country: United States Description: A satirical "mockumentary" chronicling the absurd comeback tour of an aging, fictional British heavy metal band.
Metadata: {'director': 'Rob Reiner', 'imdb_rating': '7.9/10', 'spine': '12', 'country': 'United States', 'brief_

# 6. Use upsert to fix the error on record 43

In [36]:

typo = "Encountar"
correction = "Encounter"

# grab record 43 with the typo
existing_record = chroma_collection.get(ids="43")

current_document = existing_record["documents"][0]
current_metadata = existing_record["metadatas"][0]

# replace typo in the document
updated_document = current_document.replace(typo, correction)

# fix typo in the metadata title field 
current_metadata["title_year"] = current_metadata["title_year"].replace(typo, correction)

# use upsert to replace the entire record with the corrected values
chroma_collection.upsert(
    ids="43",
    documents=[updated_document],
    metadatas=[current_metadata]
)


# 7. Use get to verify the upsert was successful

In [35]:
print(chroma_collection.get(ids="43"))

{'ids': ['43'], 'embeddings': None, 'documents': ['Spine: 76 Title: Brief Encounter (1945) Director: David Lean Country: United Kingdom Description: A married, middle-class woman and a doctor meet by chance at a railway station and begin a quiet but profound affair.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'year': '1945.0', 'title_year': 'Brief Encounter (1945)', '4k_release': 'False', 'brief_description': 'A married, middle-class woman and a doctor meet by chance at a railway station and begin a quiet but profound affair.', 'imdb_rating': '8.0/10', 'source': 'csv', 'director': 'David Lean', 'country': 'United Kingdom', 'count_sort': '76', 'spine': '76', 'rotten_tomatoes': '91%'}]}


# 8. Delete id 1

In [42]:
chroma_collection.delete(ids="1")

{'deleted': 1}

# 9. Process Five Movies.pdf using LangChain's text splitter and pypdf's reader

In [44]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

reader = PdfReader("./dataset/Five Movies.pdf")
pdf_text = ""
for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

# split the text using LangChain's text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)
chunks = text_splitter.split_text(pdf_text)

# determine the starting id
start_id = chroma_collection.count()

documents = []
metadatas = []
ids = []


# 10. Add processed document to the collection

In [45]:
# add metadata 
for i, chunk in enumerate(chunks):
    documents.append(chunk)
    metadatas.append({"source": "pdf"}) 
    ids.append(str(start_id + i))

# add processed documents to the existing collection
chroma_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# 11. Query the whole collection
- "What can you tell me about Twelve Angry Men?"
- "Which of these movies were released before 1950?"

In [46]:
query = "What can you tell me about Twelve Angry Men?"
query_results = chroma_collection.query(
    query_texts=[query],
    n_results=10,
    include=["documents", "metadatas", "embeddings", "distances"]
    )

for idx, doc in enumerate(query_results['documents'][0]):
    print(f"Result {idx+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {query_results['metadatas'][0][idx]}")
    print(f"Distance: {query_results['distances'][0][idx]}")
    print("\n")



Result 1:
Document: Spine: 591 Title: 12 Angry Men (1957) Director: Sidney Lumet Country: United States Description: A jury of 12 men must deliberate the case of a young man accused of murdering his father.
Metadata: {'year': '1957.0', 'director': 'Sidney Lumet', 'country': 'United States', '4k_release': 'False', 'count_sort': '591', 'imdb_rating': '8.9/10', 'source': 'csv', 'spine': '591', 'brief_description': 'A jury of 12 men must deliberate the case of a young man accused of murdering his father.', 'rotten_tomatoes': '100%', 'title_year': '12 Angry Men (1957)'}
Distance: 1.0636820793151855


Result 2:
Document:  Production history of 12 Angry Men, from teleplay to big-screen classic 
 Archival interviews with director Sidney Lumet 
 New interview with screenwriter Walter Bernstein about Lumet 
 New interview with Simon about writer Reginald Rose 
 Tragedy in a Temporary Town (1956), a teleplay directed by Lumet and written by 
Rose 
 New interview with cinematographer John Ba

In [47]:
query = "Which of these movies were released before 1950?"
query_results = chroma_collection.query(
    query_texts=[query],
    n_results=10,
    include=["documents", "metadatas", "embeddings", "distances"]
    )

for idx, doc in enumerate(query_results['documents'][0]):
    print(f"Result {idx+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {query_results['metadatas'][0][idx]}")
    print(f"Distance: {query_results['distances'][0][idx]}")
    print("\n")


Result 1:
Document: Spine: 66 Title: The Orphic Trilogy (Box Set) Director: Jean Cocteau Country: France Description: A box set collection of three surrealist films: The Blood of a Poet (1930), Orpheus (1950), and Testament of Orpheus (1960).
Metadata: {'imdb_rating': 'nan', 'brief_description': 'A box set collection of three surrealist films: The Blood of a Poet (1930), Orpheus (1950), and Testament of Orpheus (1960).', 'spine': '66', 'director': 'Jean Cocteau', 'count_sort': '66', '4k_release': 'False', 'year': 'nan', 'rotten_tomatoes': 'nan', 'title_year': 'The Orphic Trilogy (Box Set)', 'source': 'csv', 'country': 'France'}
Distance: 1.1719937324523926


Result 2:
Document: Spine: 79 Title: W. C. Fields—Six Short Films (1933–1937) Director: Various Country: United States Description: A collection of W. C. Fields's hilarious and influential short films.
Metadata: {'spine': '79', 'country': 'United States', 'imdb_rating': '7.7/10', '4k_release': 'False', 'year': 'nan', 'rotten_tomato

# 12. Use ollama.Client() to construct a function that performs RAG

In [54]:
from ollama import Client
client = Client(host='http://localhost:11434')

def call_llm(query, retrieved_documents, model="llama3.2:3b"):
    all_info = "\n\n".join(retrieved_documents)

    messages = [
        {
            "role": "system",
            "content": '''Your users are asking questions about information contained in the Five Movies file.
If you do not know the answer, please say I do not know. 
            Answer the user's question using only this information. If you do not know the answer, please say I do not know.'''
        },
        {"role": "user", "content": f"Question: {query}. \n Information: {all_info}"}
    ]
    
    response = client.chat(
        model=model,
        messages=messages,
    )
    content = response['message']['content']
    return content

# 13. Use the function to query the model with the previous two questions

In [56]:
query = "What can you tell me about Twelve Angry Men?"
q_result=chroma_collection.query(query_texts=[query], n_results=10)
vector_db_results=q_result['documents'][0]
llm_response = call_llm(query, vector_db_results)
print(llm_response)

I can tell you that "12 Angry Men" is a 1957 film directed by Sidney Lumet, where a jury of 12 men must deliberate the case of a young man accused of murdering his father.


In [57]:
query = "Which of these movies were released before 1950?"
q_result=chroma_collection.query(query_texts=[query], n_results=10)
vector_db_results=q_result['documents'][0]
llm_response = call_llm(query, vector_db_results)
print(llm_response)

The movies released before 1950 are:

* The Blood of a Poet (1930)
* Variety Lights (1950)

There is only one film released before 1950, Variety Lights is not released before 1950 so only one is correct.
